#鉅亨

In [1]:
#################################################################################
# 更改
# stock_id = '0000'
# glob_url = https://www.cnyes.com/search/news?keyword=股票中文名稱
 ################################################################################

stock_id = "2385"
url = 'https://www.cnyes.com/search/news?keyword=%E7%BE%A4%E5%85%89'

In [2]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By


driver = webdriver.Chrome()
driver.get(url)
news_url_l = []

for i in range(30):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

element = driver.find_elements(By.XPATH, '//a[contains(@class, "jsx-1986041679") and contains(@class, "news")]')

for e in element:
    # 獲取該元素的 href 屬性
    href = e.get_attribute('href')
    news_url_l.append(href)
    print("目標元素的網址:", href)

driver.quit()

In [18]:
news_url_l2 = news_url_l[:]

In [19]:
import time
import os
from bs4 import BeautifulSoup
import requests
from datetime import datetime
import re

content = ''
date_last = ''
article = ''

for index, link in enumerate(news_url_l2):
    # 請求網頁內容
    response = requests.get(link)
    # 解析網頁內容
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 找到所有的 <div> 元素並選擇 class="summary" 的元素
    # elements = soup.find_all('li')
    

    try:
        # 找到時間元素
        time_text = soup.find('p', class_='a1wfawlv').text.strip()
        match = re.search(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}', time_text)
    
        if match:
            time_string = match.group()
            time_datetime = datetime.strptime(time_string, '%Y-%m-%d %H:%M')
            formatted_date = time_datetime.strftime('%Y%m%d')
        
        if date_last != formatted_date:
            article = ''
        date_last = formatted_date
        
        p = soup.find('main', class_="c1tt5pk2")
        contents = p.find_all('p')
        
        for content in contents:
            article = article + content.text.strip()
        
        folder_path = "stock_news/"+stock_id+"/"+stock_id+"_news_cnyes/"
        file_name = stock_id+"n_" + formatted_date
        if not os.path.exists(os.path.dirname(folder_path)):
            print("建立資料夾" + folder_path)
            os.makedirs(os.path.dirname(folder_path))
        
        with open(folder_path + file_name, "w" , encoding='UTF-8') as f:
            f.write(article)      
            
        print(f'[{index+1}/{len(news_url_l2)}], Date:{formatted_date}, link:{link}')
        # print('日期', content + "寫入", folder_path + file_name)
            
    except Exception as e:
        print("爬取失敗", e)
        print('link', link)
        pass
    time.sleep(1)


建立資料夾stock_news/2385/2385_news_cnyes/
[1/211], Date:20240401, link:https://news.cnyes.com/news/id/5509415
[2/211], Date:20240329, link:https://news.cnyes.com/news/id/5508421
[3/211], Date:20240328, link:https://news.cnyes.com/news/id/5507470
[4/211], Date:20240327, link:https://news.cnyes.com/news/id/5505013
[5/211], Date:20240325, link:https://news.cnyes.com/news/id/5502383
[6/211], Date:20240325, link:https://news.cnyes.com/news/id/5501567
[7/211], Date:20240322, link:https://news.cnyes.com/news/id/5500591
[8/211], Date:20240322, link:https://news.cnyes.com/news/id/5500197
[9/211], Date:20240320, link:https://news.cnyes.com/news/id/5497136
[10/211], Date:20240318, link:https://news.cnyes.com/news/id/5494729
[11/211], Date:20240318, link:https://news.cnyes.com/news/id/5494728
[12/211], Date:20240315, link:https://news.cnyes.com/news/id/5491805
[13/211], Date:20240315, link:https://news.cnyes.com/news/id/5491538
[14/211], Date:20240314, link:https://news.cnyes.com/news/id/5491185
[15/2

In [20]:
# import re
# from datetime import datetime
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# import os
# 
# 
# #設定重新執行的次數
# max_retries = 3
# date_last = ''
# p = ''
# formatted_date = ''
# 
# def load_website(link):
#     global p
#     retries = 0
#     while retries < max_retries:
#         try:
#             driver = webdriver.Chrome()
#             driver.get(link)
#             
#             # time
#             time_element = driver.find_element(By.XPATH, "//p[@class='a1wfawlv']")
#             time_text = time_element.text
#             match = re.search(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}', time_text)
#     
#             if match:
#                 time_string = match.group()
#                 time_datetime = datetime.strptime(time_string, '%Y-%m-%d %H:%M')
#                 formatted_time = time_datetime.strftime('%Y%m%d')
#             print("時間：" + formatted_time)
#             
#             # 文章內容
#             if date_last != formatted_date:
#                 p = ''
#                 
#             paragraphs = driver.find_elements(By.XPATH, "//main[@id='article-container']//p")
#             
#             for paragraph in paragraphs:
#                 text = paragraph.text
#                 p = p + text
#             
#             folder_path = "stock_news/"+stock_id+"/"+stock_id+"_news/"
#             file_name = stock_id+"n_" + formatted_time
#             if not os.path.exists(os.path.dirname(folder_path)):
#                 print("建立資料夾" + folder_path)
#                 os.makedirs(os.path.dirname(folder_path))
#             
#             with open(folder_path + file_name, "w", encoding='UTF-8') as f:
#                 f.write(p)
#             
#             driver.quit()
#             return
#         
#         except Exception as e:
#             retries += 1
#             print(f"載入失敗{retries}次")
#             print(f"錯誤信息：{e}")
#             driver.quit()
#     
#     print(f"連續 {max_retries} 次載入失敗，跳過該網站")
#     time.sleep(1)
# 
# # 要加載的網站列表
# for index, link in enumerate(news_url_l2):
#     print(f'[{index+1}/{len(news_url_l2)}], link:{link}')
#     load_website(link)


In [21]:
# driver = webdriver.Chrome()
# driver.get('https://news.cnyes.com/news/id/5503840')
# 
# # time
# time_element = driver.find_element(By.XPATH, "//p[@class='a1wfawlv']")
# time_text = time_element.text
# match = re.search(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}', time_text)
# 
# if match:
#     time_string = match.group()
#     time_datetime = datetime.strptime(time_string, '%Y-%m-%d %H:%M')
#     formatted_time = time_datetime.strftime('%Y%m%d')
# print("時間：" + formatted_time)
# 
# # 文章內容
# if date_last != formatted_date:
#     p = ''
#     
# paragraphs = driver.find_elements(By.XPATH, "//main[@id='article-container']//p")
# 
# for paragraph in paragraphs:
#     text = paragraph.text
#     p = p + text
# 
# folder_path = "stock_news/"+stock_id+"/"+stock_id+"_news/"
# file_name = stock_id+"n_" + formatted_time
# if not os.path.exists(os.path.dirname(folder_path)):
#     print("建立資料夾" + folder_path)
#     os.makedirs(os.path.dirname(folder_path))
# 
# with open(folder_path + file_name, "w") as f:
#     f.write(p)
#     
# print(p)
# 
# driver.quit()